In [1]:
import time
from labjack import ljm

# ==========================================
# HARDWARE & CALIBRATION CONSTANTS
# ==========================================
# LabJack Register for AIO5
INPUT_REGISTER = "AIN5"

# LJTick-CurrentShunt Specific Equation: mA = 8.475 * Volts
LJTCS_SLOPE = 8.475

# Sensor Level Limits (Height in feet)
H_MIN = 0.0
H_MAX = 1.96

# Expected Voltage Limits from LJTCS for a 4-20mA Signal
V_MIN = 0.472  # 4mA baseline
V_MAX = 2.360  # 20mA baseline

# Low-End Noise Cutoff (in mA)
# Forces the reading to exactly 0.0 ft if the current drops to or below this floor
ZERO_CUTOFF_MA = 4.05


def process_level_data(voltage):
    """Converts raw voltage to mA and height (ft) with a low-end zero floor."""
    # 1. Convert voltage to mA using LJTCS transfer equation
    current_mA = LJTCS_SLOPE * voltage

    # 2. Enforce zero floor condition
    if current_mA <= ZERO_CUTOFF_MA:
        height_ft = 0.0
    else:
        # 3. Linear interpolation mapping: Volts to Feet
        height_ft = H_MIN + (voltage - V_MIN) * (H_MAX - H_MIN) / (
            V_MAX - V_MIN
        )

        # Safety clamp to prevent minor over-range fluctuations past max height
        if height_ft > H_MAX:
            height_ft = H_MAX

    return current_mA, height_ft


def main():
    try:
        # Open connection to the T7
        handle = ljm.openS("T7", "ANY", "ANY")
        info = ljm.getHandleInfo(handle)
        print(f"Connected to LabJack T7 [Serial: {info[2]}]")
        print(f"Monitoring Standalone Level Sensor on {INPUT_REGISTER}")
        print("-" * 55)
        print(
            f"{'Voltage (V)':<15}{'Current (mA)':<15}{'Level (ft)':<15}{'Status':<10}"
        )
        print("-" * 55)

        # Configure the analog input channel
        ljm.eWriteName(handle, f"{INPUT_REGISTER}_NEGATIVE_CH", 199)  # Single-Ended to GND
        ljm.eWriteName(handle, f"{INPUT_REGISTER}_RANGE", 10.0)  # +/- 10V Scale to avoid clipping

        while True:
            # Read voltage from AIO5/AIN5
            voltage = ljm.eReadName(handle, INPUT_REGISTER)

            # Process data through the conversion logic
            current_mA, level_ft = process_level_data(voltage)

            # Determine loop operational status
            if level_ft == 0.0:
                status = "DRY / FLOOR"
            elif current_mA > 20.2:
                status = "OVER RANGE"
            else:
                status = "ACTIVE"

            # Print data cleanly to the terminal line
            print(
                f"{voltage:<15.4f}{current_mA:<15.4f}{level_ft:<15.4f}{status:<10}",
                end="\r",
            )

            time.sleep(0.5)

    except ljm.LJMError as e:
        print(f"\nLabJack LJM Error: {e}")
    except KeyboardInterrupt:
        print("\nTesting stopped by user.")
    finally:
        # Cleanly disconnect from hardware
        if "handle" in locals():
            ljm.close(handle)
            print("LabJack connection closed safely.")


if __name__ == "__main__":
    main()

Connected to LabJack T7 [Serial: 470042305]
Monitoring Standalone Level Sensor on AIN5
-------------------------------------------------------
Voltage (V)    Current (mA)   Level (ft)     Status    
-------------------------------------------------------
0.4730         4.0090         0.0000         DRY / FLOOR
Testing stopped by user.
LabJack connection closed safely.
